## Montando DF

In [ ]:
import torch
from torchinfo import summary
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from codecarbon import EmissionsTracker
import datetime
import sys
from typing import Union
from scipy.optimize import minimize_scalar

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from core.models.encdec_model import EncDecModel
from core.models.stacked_rnn import Stacked_RNN
from domains.previsao_ceu.dataset_module_group import SolarEfficientDatasetGroups
from domains.previsao_ceu.preprocessing import SolarPreprocessor
from domains.previsao_ceu.dataset import SolarEfficientDataset
from core.utils.experiment_manager import ExperimentManager
from core.utils.early_stopping import EarlyStopping
from auxiliary_models.brl_diffuse import BRL
from auxiliary_models.kasten_correction import Kasten_Correction
from core.loss_function.cpiloss import CPILoss
from core.loss_function.mseloss import MaskedMSELoss
from core.loss_function.pyloss import PhysicsGuidedLoss

In [ ]:
CONFIG = {
    # --- 1. PATH DA BASE DE DADOS ---
    'csv_path': 'data/pv0.csv',
    
    # --- 2. DIVISÃO DA BASE DE TREINO, TESTE E VALIDAÇÃO ---
    'split_ratios': {'train': 0.8, 'val': 0.2}, 
    'test_year':2022,

    # --- 3. PRÉ-PROCESSAMENTO (Física & Mapeamento) ---
    'preprocessing': {
        'latitude': -23.56,
        'longitude': -46.73,
        'altitude': 0,
        'timezone': 'Etc/GMT+3',
        'nominal_power': 156.0,
        'start_year': 2015,
        'features_to_scale':['temp_amb','wind_speed'],
        #'pv_power_col_csv': 'Pot_BT', # <--- AVALIAR PARA RETIRAR
        
        # DOCUMENTAÇÃO VIVA: Mapeamento "De -> Para"
        # O Preprocessor usará isso para renomear as colunas internamente.
        # Chave (Esquerda): Nome como está no CSV bruto.
        # Valor (Direita): Nome padronizado usado no código.
        'column_mapping': {
            'Pot_BT': 'target',
            'Irradiação Global horária(horizontal) kWh/m2': 'ghi',
            'Irradiação Difusa horária kWh/m2': 'dhi',
            'Irradiação Global horária(Inclinada 27°) kWh/m2': 'irrad_poa',
            'Temperatura ambiente °C': 'temp_amb',
            'Umidade Relativa %': 'humidity',
            'Velocidade média do vento m/s': 'wind_speed'
        }
    },

    # --- 4. ESTRATÉGIA DE MODELAGEM ---
    # mode: 'sky' (prevê k) ou 'power' (prevê kW normalizado)
    'prediction_mode': 'sky',
    
    # Qual variável o modelo vai prever? ('k' ou 'target')
    'target_col': ['kt', 'fracao_difusa'], 
    
    # Features de entrada
    'feature_cols': [
        "kt",'fracao_difusa',
        'cos_zenith', 'elevation', 'delta_kt', 
        'delta_fracao_difusa', 'QS', 
        'temp_amb', 'humidity', 'wind_speed'
    ],
    
    'aux_col':[
        'ghi_cs', 'cos_zenith', 
        'elevation', 'ghi_extra'
    ],

    # --- 5. ARQUITETURA E TREINO ---
    'model_type': 'Teste',
    'cell_type': 'lstm',
    'input_seq_len': 24,
    'output_seq_len': 1,
    'hidden_sizes': [300],
    'learning_rate': 0.001,
    'batch_size': 32,
    'epochs': 10000,
    'dropout': 0.1,
    'bidirectional': False,
    'use_attention': False,
    'use_feature_attention': False,
    'patience': 100,
    'use_mask':True,
    'loss_function':'physics_loss',      # "cpi_loss", "mse", physics_loss
    'physics_base_loss': 'cpi',
    'lambda_hard':1,
    'lambda_soft':1,
}

OUTPUT_ROOT = 'trained_models'
ARTIFACTS_DIR = 'artifacts'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 1. Setup
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
exp_name = f"{timestamp}_{CONFIG['model_type']}"
exp_dir = os.path.join(OUTPUT_ROOT, exp_name)
os.makedirs(exp_dir, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# 2. Leitura
#csv_path = CONFIG['data/pv0.csv']
#print(f"⏳ Lendo: {csv_path}")
df = pd.read_csv('/workspaces/Remodelacao_mestrado/data/pv0.csv')


if 'Date_Time' in df.columns:
    df['Date_Time'] = pd.to_datetime(df['Date_Time'])
    #df['Date_Time'] -=  pd.Timedelta(minutes=30)
    df = df.drop_duplicates(subset=['Date_Time'], keep='first').set_index('Date_Time').sort_index()
df = df[~df.index.duplicated(keep='first')]

# 3. Pré-processamento
pp_conf = CONFIG['preprocessing']

# Instancia passando o mapa explícito. 
# Isso garante que a padronização aconteça conforme o CONFIG acima.
preprocessor = SolarPreprocessor(
    latitude=pp_conf['latitude'], 
    longitude=pp_conf['longitude'], 
    altitude=pp_conf['altitude'],
    timezone=pp_conf['timezone'], 
    nominal_power=pp_conf['nominal_power'], 
    start_year=pp_conf['start_year'],
    cs_model = 'esra',
    features_to_scale=pp_conf['features_to_scale'],
    target_col=CONFIG['prediction_mode'], # <--- unica variavel que não vem do preprocessing
    column_mapping=pp_conf['column_mapping'],
    kasten_corr=True
)

preprocessor.fit(df)
preprocessor.save_scalers(exp_dir)
preprocessor.save_scalers(ARTIFACTS_DIR)

# O método transform usa o column_mapping para renomear as colunas
df_processed = preprocessor.transform(df)

💾 Scalers salvos em: trained_models/2026-03-19_13-23-41_Teste
💾 Scalers salvos em: artifacts
Otimizando TL para 381 dias selecionados...
Otimização concluída. Média TL: 6.28


## Treinamento

In [ ]:
# 4. Validação de Colunas
target_col = CONFIG['target_col']

for col in target_col:
    if col not in df_processed.columns:
        raise ValueError(f"❌ Coluna alvo '{target_col}' não encontrada! Verifique o column_mapping.")

available_cols = [c for c in CONFIG['feature_cols'] if c in df_processed.columns]
if len(available_cols) != len(CONFIG['feature_cols']):
    print(f"⚠️ Features ajustadas: {available_cols}")
    CONFIG['feature_cols'] = available_cols

# 5. Split Temporal (Último Ano = Teste)
#last_year = df_processed.index.year.max() <---- REMOVER FUTURAMENTE
last_year = CONFIG['test_year']
print(f"📅 Separando ano {last_year} para TESTE.")

test_df = df_processed[df_processed.index.year == last_year].copy()
dev_df = df_processed[df_processed.index.year < last_year].copy()

if dev_df.empty:
    raise ValueError("❌ Erro no Split: Dados insuficientes antes do último ano.")

# Split Treino/Validação
n_dev = len(dev_df)
train_end = int(n_dev * CONFIG['split_ratios']['train'])

train_df = dev_df.iloc[:train_end].copy()
val_df = dev_df.iloc[train_end:].copy()

print(f"📊 Divisão: Treino={len(train_df)} | Val={len(val_df)} | Teste={len(test_df)}")


#
#   --- Variação do n_future ---
#

n_future_array = [1, 3, 6, 12, 24]  # Exemplo: 1h, 3h, 6h, 12h, 24h
for n_f in n_future_array:

    CONFIG['output_seq_len'] = n_f
    CONFIG['model_type'] = f"EDLSTM_fut_{n_f}h"

    # DataLoaders
    train_dataset = SolarEfficientDataset(
        df=train_df, 
        feature_cols=CONFIG['feature_cols'], 
        target_col=CONFIG['target_col'],
        aux_col=CONFIG['aux_col'],
        n_past=CONFIG['input_seq_len'], 
        n_future=CONFIG['output_seq_len']
    )
    val_dataset = SolarEfficientDataset(
        df=val_df, 
        feature_cols=CONFIG['feature_cols'], 
        target_col=CONFIG['target_col'],
        aux_col=CONFIG['aux_col'],
        n_past=CONFIG['input_seq_len'], 
        n_future=CONFIG['output_seq_len']
    )

    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

    # 6. Modelo
    model = EncDecModel(
        input_size=len(CONFIG['feature_cols']),
        hidden_sizes=CONFIG['hidden_sizes'],
        output_seq_len=CONFIG['output_seq_len'],
        output_dim=len(CONFIG['target_col']),
        cell_type=CONFIG['cell_type'],
        bidirectional=CONFIG['bidirectional'],
        use_attention=CONFIG['use_attention'],
        use_feature_attention=CONFIG['use_feature_attention'],
        dropout_prob=CONFIG['dropout']
    ).to(DEVICE)

    # modificar o criterion muda a função de erro implementado a CPILoss
    if CONFIG['loss_function'] == 'mse':
        criterion = MaskedMSELoss()
    elif CONFIG['loss_function'] == 'cpi_loss':
        criterion = CPILoss()

    optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
    early_stopping = EarlyStopping(patience=CONFIG['patience'], verbose=True, path=os.path.join(exp_dir, 'best_model.pt'))

    # resumo do modelo
    sample_input_size = (CONFIG['batch_size'], CONFIG['input_seq_len'], len(CONFIG['feature_cols']))
    print("\n📊 Resumo do Modelo:")
    display(summary(model, input_size=sample_input_size, device=DEVICE))

    # 7. Treino
    print("🔥 Iniciando épocas...")
    train_losses, val_losses = [], []
    start_time = time.time()

    for epoch in range(CONFIG['epochs']):
        model.train()
        batch_losses = []
        for x, y, mask, aux in train_loader:
            x, y, mask, aux = x.to(DEVICE), y.to(DEVICE), mask.to(DEVICE), aux.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            active_mask = mask if CONFIG['use_mask'] else None
            loss = criterion(out, y, mask=active_mask)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        
        avg_train = np.mean(batch_losses)
        train_losses.append(avg_train)

        model.eval()
        val_batch_losses = []
        with torch.no_grad():
            for x, y, mask, aux in val_loader:
                x, y, mask, aux = x.to(DEVICE), y.to(DEVICE), mask.to(DEVICE), aux.to(DEVICE)
                out = model(x)
                active_mask = mask if CONFIG['use_mask'] else None

                loss = criterion(out, y, aux_future=aux, mask=active_mask)
                
                val_batch_losses.append(loss.item())
        
        avg_val = np.mean(val_batch_losses)
        val_losses.append(avg_val)
        
        print(f"Epoch {epoch+1} | Train: {avg_train:.6f} | Val: {avg_val:.6f}")
        
        early_stopping(avg_val, model)
        if early_stopping.early_stop:
            print("🛑 Early stopping.")
            break
